# TheLook 쇼핑몰 비즈니스 용어집 구축 및 관리

이 노트북은 대화형 분석 에이전트가 비즈니스 사용자의 다양한 자연어 질문(예: "VIP 고객", "이탈 고객", "마진율" 등)을 올바른 BigQuery SQL 쿼리로 정확하게 변환할 수 있도록 돕는 **Knowledge Catalog 비즈니스 용어집** 등록 가이드입니다.

GCS 버킷에 사전 업로드된 파일에 정의된 비즈니스 용어 정의와 **SQL 매핑 aspect**를 로드하여, `gcloud` CLI 및 Knowledge Catalog API를 통해 용어집, 카테고리, 용어를 생성하고 관계를 연동하는 전체 파이프라인을 실행합니다.

### 학습 목표
1. **Knowledge Catalog Glossary 구조 이해**: 비즈니스 용어집(Glossary)과 용어(Term), 카테고리(Category) 간의 상하 계층 아키텍처를 이해합니다.
2. **커스텀 Aspect Type 생성 및 활용**: 비즈니스 언어를 물리 데이터(SQL 쿼리)로 매핑시키기 위한 사용자 정의 메타데이터 속성(`sql-mapping`)을 선언하고 채우는 기법을 배웁니다.
3. **용어-자산 관계 링킹**: 생성된 논리 비즈니스 용어들을 실제 BigQuery 물리 테이블 및 특정 컬럼들과 연결(Relationship)하는 파이프라인을 구현합니다.

### 작업 파이프라인

1. **환경 초기화**: GCP 프로젝트 설정 및 REST 호출을 위한 OAuth2 토큰을 획득합니다.
2. **API 호출 공통 함수 정의**: API 호출 편의성을 위해 Knowledge Catalog 및 DataScan REST API용 헬퍼 요청 함수를 정의합니다.
3. **기존 용어집 정리**: 재초기화(`REINITIALIZE=True`) 시 기존에 존재하는 용어집 및 커스텀 Aspect Type을 삭제하여 초기 상태로 만듭니다.
4. **용어집 및 Aspect Type 생성**: Knowledge Catalog 비즈니스 용어집과 용어에 매핑할 커스텀 SQL 매핑 Aspect Type을 생성합니다.
5. **비즈니스 용어 로드**: GCS 버킷에 적재된 CSV/JSON 파일로부터 비즈니스 용어 정의와 SQL 매핑 설정 메타데이터를 파싱합니다.
6. **비즈니스 카테고리 및 용어 생성**: 정의된 테이블 및 규칙에 따라 용어집 내에 카테고리와 용어 리소스 구조를 생성합니다.
7. **리소스 연동 수립**: 생성된 비즈니스 용어를 관련 BigQuery 테이블 및 컬럼과 논리적으로 매핑 연동합니다.
8. **비즈니스 용어 Aspect 설정**: 각 용어에 개요(Overview) 및 SQL 매핑 Aspect를 부여합니다.
9. **최종 결과 검증**: `gcloud` 명령어와 Knowledge Catalog API를 통해 카테고리 및 용어 연동 결과를 최종 조회합니다.

## Step 1: 초기 환경 설정 및 REST API 설정

In [ ]:
import json
import ssl
import urllib.request
import urllib.error
import time
import subprocess
import google.auth
from google.auth.transport.requests import Request

# 1. 설정 정보 정의
GLOSSARY_ID = "thelook-glossary"
LOCATION = "us-central1"  # 빅쿼리 데이터셋 위치(us-central1 리전)와 일치시켜 리전간 연동(EntryLinks) 문제 해결
REINITIALIZE = False           # True 설정 시 기존 용어집 자원을 전체 삭제 후 초기화 구축합니다.

# GCP 프로젝트 ID 조회 및 Access Token 발급
credentials, PROJECT_ID = google.auth.default()
credentials.refresh(Request())
ACCESS_TOKEN = credentials.token

ssl_context = ssl._create_unverified_context()

# REST API를 사용하여 프로젝트 번호(Project Number) 조회
req = urllib.request.Request(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}",
    headers={"Authorization": f"Bearer {ACCESS_TOKEN}"}
)
with urllib.request.urlopen(req, context=ssl_context) as response:
    project_info = json.loads(response.read())
    PROJECT_NUMBER = project_info["projectNumber"]

print(f"GCP Project ID: {PROJECT_ID} (Project Number: {PROJECT_NUMBER})")

## Step 2: API 호출 공통 함수 정의

In [ ]:
existing_categories = None

# ==========================================
# [공통] 지수 백오프 및 재시도 내장 REST 요청 헬퍼
# ==========================================
def send_rest_request_with_retry(url, method="POST", body=None, max_attempts=5, ignore_409=True, retry_on_403_404=False):
    """
    백오프 및 특정 HTTP 에러 대응 로직을 중앙 집중화한 REST API 요청 공통 함수입니다.
    """
    headers = {
        "Authorization": f"Bearer {ACCESS_TOKEN}",
        "Content-Type": "application/json"
    }
    data = json.dumps(body).encode("utf-8") if body else None
    backoff = 2
    
    for attempt in range(max_attempts):
        req = urllib.request.Request(url, data=data, headers=headers, method=method)
        try:
            with urllib.request.urlopen(req, context=ssl_context) as res:
                return True
        except urllib.error.HTTPError as e:
            # 409 Conflict: 이미 존재하는 리소스인 경우 성공으로 간주
            if e.code == 409 and ignore_409:
                return True
            # 429 Rate Limit: 지수 백오프 대기 후 재시도
            elif e.code == 429:
                print(f"    [WAIT] 호출 제한(429)으로 인해 대기 후 재시도합니다. (시도 {attempt+1}/{max_attempts})...")
                time.sleep(backoff)
                backoff *= 2
            # 403/404: 인덱싱 미반영 대기 (Aspect 주입 단계 등에서 활성화 가능)
            elif e.code in (403, 404) and retry_on_403_404:
                print(f"    [WAIT] 인덱싱 반영 대기 중... 5초 후 재시도합니다. (시도 {attempt+1}/{max_attempts})")
                time.sleep(5)
            else:
                print(f"    [FAIL] API 호출 실패: {e.code} - {e.read().decode('utf-8')}")
                return False
                
    print(f"    [FAIL] API 호출 실패 (최대 시도 횟수 초과)")
    return False


# ==========================================
# 개별 리소스 관리 및 연동 함수
# ==========================================

# 1. 카테고리 생성 (gcloud CLI 기반)
def get_or_create_category(cat_id, display_name, parent_path):
    global existing_categories
    if existing_categories is None:
        existing_categories = set()
        list_cats = subprocess.run([
            "gcloud", "dataplex", "glossaries", "categories", "list",
            f"--glossary={GLOSSARY_ID}", f"--location={LOCATION}", f"--project={PROJECT_ID}",
            "--format=value(name)"
        ], capture_output=True, text=True)
        if list_cats.returncode == 0 and list_cats.stdout.strip():
            for cat_path in list_cats.stdout.strip().split("\n"):
                existing_categories.add(cat_path.split("/")[-1])
        print(f"기존 등록 카테고리 캐싱 완료 (기존 카테고리: {len(existing_categories)}개)")

    if cat_id not in existing_categories:
        print(f"카테고리 생성 중: {cat_id} ({display_name})...")
        subprocess.run([
            "gcloud", "dataplex", "glossaries", "categories", "create", cat_id,
            f"--glossary={GLOSSARY_ID}", f"--location={LOCATION}", f"--project={PROJECT_ID}",
            f"--parent={parent_path}",
            f"--display-name={display_name}"
        ], check=True)
        existing_categories.add(cat_id)
        print(f"  카테고리 생성 성공: {cat_id}. 인덱싱 대기 (3초)...")
        time.sleep(3)
    return f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}/categories/{cat_id}"

# 2. 용어 생성
def create_glossary_term(term_id, display_name, description, parent_path):
    url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}/terms?termId={term_id}"
    body = {"displayName": display_name, "description": description, "parent": parent_path}
    return send_rest_request_with_retry(url, method="POST", body=body)

# 3. 테이블 연동
def link_term_to_table(term_entry_path, bq_entry_path, link_id):
    url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@bigquery/entryLinks?entry_link_id={link_id}"
    body = {
        "entryLinkType": "projects/dataplex-types/locations/global/entryLinkTypes/definition",
        "entryReferences": [
            {"name": bq_entry_path, "type": "SOURCE"},
            {"name": term_entry_path, "type": "TARGET"}
        ]
    }
    return send_rest_request_with_retry(url, method="POST", body=body)

# 4. 연관 용어 상호 연동
def link_related_terms(term_1_path, term_2_path, link_id):
    url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@dataplex/entryLinks?entry_link_id={link_id}"
    body = {
        "entryLinkType": "projects/dataplex-types/locations/global/entryLinkTypes/related",
        "entryReferences": [
            {"name": term_1_path, "type": "UNSPECIFIED"},
            {"name": term_2_path, "type": "UNSPECIFIED"}
        ]
    }
    return send_rest_request_with_retry(url, method="POST", body=body)

# 5. 동의어 상호 연동
def link_synonym_terms(term_1_path, term_2_path, link_id):
    url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@dataplex/entryLinks?entry_link_id={link_id}"
    body = {
        "entryLinkType": "projects/dataplex-types/locations/global/entryLinkTypes/synonym",
        "entryReferences": [
            {"name": term_1_path, "type": "UNSPECIFIED"},
            {"name": term_2_path, "type": "UNSPECIFIED"}
        ]
    }
    return send_rest_request_with_retry(url, method="POST", body=body)

# 6. 개요 Aspect 등록
def set_term_overview(term_id, overview_text):
    term_entry_name = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@dataplex/entries/projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}/terms/{term_id}"
    url = f"https://dataplex.googleapis.com/v1/{term_entry_name}?updateMask=aspects"
    body = {
        "aspects": {
            "dataplex-types.global.overview": {
                "aspectType": "projects/dataplex-types/locations/global/aspectTypes/overview",
                "data": {"content": overview_text, "links": [], "contentType": "MARKDOWN"}
            }
        }
    }
    return send_rest_request_with_retry(url, method="PATCH", body=body, max_attempts=10, retry_on_403_404=True)

# 7. SQL 매핑 Aspect 등록
def set_term_sql_mapping(term_id, mapping_data):
    term_entry_name = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@dataplex/entries/projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}/terms/{term_id}"
    url = f"https://dataplex.googleapis.com/v1/{term_entry_name}?updateMask=aspects"
    body = {
        "aspects": {
            f"{PROJECT_ID}.{LOCATION}.sql-mapping": {
                "aspectType": f"projects/{PROJECT_ID}/locations/{LOCATION}/aspectTypes/sql-mapping",
                "data": mapping_data
            }
        }
    }
    return send_rest_request_with_retry(url, method="PATCH", body=body, max_attempts=10, retry_on_403_404=True)

## Step 3: 기존 비즈니스 용어집 정리

In [ ]:
def cleanup_glossary_if_exists():
    describe_glossary = subprocess.run(
        ["gcloud", "dataplex", "glossaries", "describe", GLOSSARY_ID, f"--location={LOCATION}", f"--project={PROJECT_ID}"],
        capture_output=True, text=True
    )
    if describe_glossary.returncode != 0:
        print(f"Glossary '{GLOSSARY_ID}'가 존재하지 않습니다. 새로 생성합니다.")
        return

    print(f"기존 Glossary '{GLOSSARY_ID}' 발견. 재생성을 위해 하위 요소를 역순으로 정리합니다...")

    # A. 모든 용어(Terms) 삭제
    terms_proc = subprocess.run([
        "gcloud", "dataplex", "glossaries", "terms", "list",
        f"--glossary={GLOSSARY_ID}", f"--location={LOCATION}", f"--project={PROJECT_ID}",
        "--format=value(name)"
    ], capture_output=True, text=True)
    
    if terms_proc.stdout.strip():
        for term_path in terms_proc.stdout.strip().split("\n"):
            term_id = term_path.split("/")[-1]
            print(f"  용어 삭제 중: {term_id}...")
            subprocess.run([
                "gcloud", "dataplex", "glossaries", "terms", "delete", term_id,
                f"--glossary={GLOSSARY_ID}", f"--location={LOCATION}", f"--project={PROJECT_ID}", "--quiet"
            ], check=True)

    # B. 모든 카테고리(Categories) 삭제
    cats_proc = subprocess.run([
        "gcloud", "dataplex", "glossaries", "categories", "list",
        f"--glossary={GLOSSARY_ID}", f"--location={LOCATION}", f"--project={PROJECT_ID}",
        "--format=value(name)"
    ], capture_output=True, text=True)
    
    if cats_proc.stdout.strip():
        cats = cats_proc.stdout.strip().split("\n")
        cats.sort(key=len, reverse=True)
        for cat_path in cats:
            cat_id = cat_path.split("/")[-1]
            print(f"  카테고리 삭제 중: {cat_id}...")
            subprocess.run([
                "gcloud", "dataplex", "glossaries", "categories", "delete", cat_id,
                f"--glossary={GLOSSARY_ID}", f"--location={LOCATION}", f"--project={PROJECT_ID}", "--quiet"
            ], check=True)

    # C. 용어집(Glossary) 본체 삭제
    print(f"용어집 본체 삭제 중: {GLOSSARY_ID}...")
    subprocess.run([
        "gcloud", "dataplex", "glossaries", "delete", GLOSSARY_ID,
        f"--location={LOCATION}", f"--project={PROJECT_ID}", "--quiet"
    ], check=True)
    print("기존 용어집 정리 완료. 안정적인 카탈로그 동기화를 위해 10초간 대기합니다...")
    time.sleep(10)

# REINITIALIZE 플래그에 따라 작동 선택
if REINITIALIZE:
    cleanup_glossary_if_exists()
else:
    print("REINITIALIZE 플래그가 False입니다. 기존 용어집 정리를 건너뛰고 증분 모드로 진행합니다.")

## Step 4: 신규 비즈니스 용어집 생성

In [ ]:
# Aspect Type 존재 확인 및 생성
def ensure_sql_mapping_aspect_type():
    describe_aspect = subprocess.run([
        "gcloud", "dataplex", "aspect-types", "describe", "sql-mapping",
        f"--location={LOCATION}", f"--project={PROJECT_ID}"
    ], capture_output=True, text=True)
    
    if describe_aspect.returncode != 0:
        print("\nCustom Aspect Type 'sql-mapping'이 존재하지 않습니다. 생성을 진행합니다...")
        import os
        from google.cloud import storage
        
        # aspect_sql_mapping.json 파일 경로 확인 (노트북 위치 기준)
        schema_path = "../resources/aspect_sql_mapping.json"
        if not os.path.exists(schema_path):
            schema_path = "analytics/resources/aspect_sql_mapping.json"
            if not os.path.exists(schema_path):
                schema_path = "resources/aspect_sql_mapping.json"
                
        # 로컬 파일이 없으면 GCS에서 다운로드
        if not os.path.exists(schema_path):
            print(f"로컬에서 스키마 파일을 찾지 못했습니다. GCS 버킷(gs://metadata-resources-{PROJECT_ID}/resources/aspect_sql_mapping.json)에서 다운로드합니다...")
            storage_client = storage.Client(project=PROJECT_ID)
            bucket = storage_client.bucket(f"metadata-resources-{PROJECT_ID}")
            blob = bucket.blob("resources/aspect_sql_mapping.json")
            if blob.exists():
                schema_path = "aspect_sql_mapping.json"
                blob.download_to_filename(schema_path)
                print(f"  GCS로부터 스키마 다운로드 완료: {schema_path}")
            else:
                raise FileNotFoundError("로컬 및 GCS 모두에서 aspect_sql_mapping.json 파일을 찾을 수 없습니다. 인프라 배포(Terraform) 상태를 확인하세요.")
                
        print(f"스키마 파일 로드 경로: {schema_path}")
        subprocess.run([
            "gcloud", "dataplex", "aspect-types", "create", "sql-mapping",
            f"--location={LOCATION}", f"--project={PROJECT_ID}",
            f"--metadata-template-file-name={schema_path}",
            "--display-name=SQL Mapping Ruleset",
            "--description=Ruleset for BigQuery Agent Text-to-SQL mapping."
        ], check=True)
        print("Custom Aspect Type 'sql-mapping' 생성 완료. 인덱싱을 위해 5초간 대기합니다...")
        time.sleep(5)
    else:
        print("Custom Aspect Type 'sql-mapping'이 이미 존재합니다.")

# Glossary 존재 확인 헬퍼 함수
def check_glossary_exists():
    describe_glossary = subprocess.run([
        "gcloud", "dataplex", "glossaries", "describe", GLOSSARY_ID,
        f"--location={LOCATION}", f"--project={PROJECT_ID}"
    ], capture_output=True, text=True)
    return describe_glossary.returncode == 0

# 먼저 Aspect Type 존재 확인 및 생성 수행
ensure_sql_mapping_aspect_type()

# 존재하지 않을 경우에만 생성 진행
if not check_glossary_exists():
    print(f"\n새로운 Glossary '{GLOSSARY_ID}'를 생성합니다...")
    subprocess.run([
        "gcloud", "dataplex", "glossaries", "create", GLOSSARY_ID,
        f"--location={LOCATION}", f"--project={PROJECT_ID}",
        "--description=TheLook Ecommerce Business Glossary with Taxonomy",
        "--display-name=TheLook Glossary"
    ], check=True)
    print("용어집 생성 완료. 카탈로그 인덱싱 동기화를 위해 15초간 대기합니다...")
    time.sleep(15)
else:
    print(f"Glossary '{GLOSSARY_ID}'가 이미 존재합니다. 생성을 건너뜁니다.")

## Step 5: 비즈니스 용어 정의 데이터셋 로드

In [ ]:
# GCS 버킷 및 파일 경로 정의
RESOURCE_BUCKET = f"metadata-resources-{PROJECT_ID}"
GCS_BLOB_PATH = "resources/business_glossary.json"
LOCAL_GLOSSARY_PATH = "../resources/business_glossary.json"

import os
import json
from google.cloud import storage

# 로컬 파일이 존재하는 경우 우선적으로 로드 (로컬 개발 및 수정 테스트 시 유용)
if os.path.exists(LOCAL_GLOSSARY_PATH):
    print(f"로컬 파일 발견. 로컬 경로에서 용어집을 로드합니다: {LOCAL_GLOSSARY_PATH}")
    with open(LOCAL_GLOSSARY_PATH, "r", encoding="utf-8") as f:
        glossary_data = json.load(f)

# 로컬 파일이 없는 경우 (예: Colab 원격 실습 환경) -> GCS에서 다운로드하여 로드
else:
    print(f"로컬 파일이 없습니다. GCS 버킷(gs://{RESOURCE_BUCKET}/{GCS_BLOB_PATH})에서 데이터를 다운로드합니다...")
    # Storage 클라이언트 초기화
    storage_client = storage.Client(project=PROJECT_ID)
    bucket = storage_client.bucket(RESOURCE_BUCKET)
    blob = bucket.blob(GCS_BLOB_PATH)
    
    if blob.exists():
        glossary_data = json.loads(blob.download_as_text())
        print("  GCS로부터 데이터 로드 완료.")
    else:
        raise FileNotFoundError(f"로컬 및 GCS(gs://{RESOURCE_BUCKET}/{GCS_BLOB_PATH}) 모두에서 비즈니스 용어집 파일을 찾을 수 없습니다. 인프라 배포(Terraform) 상태를 확인하세요.")

print(f"총 {len(glossary_data)}개의 비즈니스 용어 정의가 준비되었습니다.")

## Step 6: 비즈니스 카테고리 및 용어 생성

In [ ]:
# 5. 계층 구조에 따른 카테고리 및 용어 생성
glossary_parent = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}"

# 기존에 등록된 모든 용어 리스트를 조회하여 세트에 저장
existing_terms = set()
list_terms = subprocess.run([
    "gcloud", "dataplex", "glossaries", "terms", "list",
    f"--glossary={GLOSSARY_ID}", f"--location={LOCATION}", f"--project={PROJECT_ID}",
    "--format=value(name)"
], capture_output=True, text=True)

if list_terms.returncode == 0 and list_terms.stdout.strip():
    for term_path in list_terms.stdout.strip().split("\n"):
        term_id = term_path.split("/")[-1]
        existing_terms.add(term_id)
print(f"기존 등록 용어 조회 완료 (기존 용어: {len(existing_terms)}개)")

# ----------------------------------------------------
# 모든 카테고리, 메인 용어 및 동의어 생성
# ----------------------------------------------------
print("\n--- 모든 카테고리 및 용어 생성 시작 ---")
new_terms_created = False

for entry in glossary_data:
    cat_id = entry["category_id"]
    cat_name = entry["category"]
    cat_path = get_or_create_category(cat_id, cat_name, glossary_parent)
    
    sub_cat_id = entry["sub_category_id"]
    sub_cat_name = entry["sub_category"]
    sub_cat_path = get_or_create_category(sub_cat_id, sub_cat_name, cat_path)
    
    term_name = entry["term"]
    term_id = entry["term_id"]
    description = entry.get("description", "")
    synonyms = entry.get("synonyms", [])
    
    # 1. 메인 용어 생성
    if term_id not in existing_terms:
        print(f"메인 용어 등록 중 (REST API): {term_id} ({term_name}) under {sub_cat_id}...")
        if create_glossary_term(term_id, term_name, description, sub_cat_path):
            existing_terms.add(term_id)
            new_terms_created = True
            
    # 2. 동의어 생성
    for idx, syn_name in enumerate(synonyms):
        syn_term_id = f"syn-{term_id}-{idx+1}"
        if syn_term_id not in existing_terms:
            syn_desc = f"[{term_name}의 동의어] {description}"
            print(f"  └─ 동의어 등록 중 (REST API): {syn_term_id} ({syn_name})...")
            if create_glossary_term(syn_term_id, syn_name, syn_desc, sub_cat_path):
                existing_terms.add(syn_term_id)
                new_terms_created = True

# 새롭게 생성된 용어가 있는 경우에만 인덱싱 대기를 위해 일시 대기
if new_terms_created:
    print("\n신규 용어가 생성되었습니다. Knowledge Catalog 검색 인덱싱 반영을 위해 15초간 대기합니다...")
    time.sleep(15)
else:
    print("\n새로 생성된 용어가 없어 대기 없이 즉시 진행합니다.")

## Step 7: 리소스 연동 수립

In [ ]:
# 안정적인 연동 수립을 위해 진입 전 대기
print("\n용어 생성 완료. 관계 연동 수립 진입 전 인덱싱 완료를 위해 10초간 대기합니다...")
time.sleep(10)

# 생성 완료된 리소스들 간의 연동 링크(EntryLinks) 설정
print("\n--- [데이터 자산 및 용어 간 연동 수립 (EntryLinks)] ---")
processed_links = set()

for entry in glossary_data:
    term_id = entry["term_id"]
    term_name = entry["term"]
    term_entry_path = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@dataplex/entries/projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}/terms/{term_id}"
    
    # A. 테이블 데이터 자산 연동 (type = definition)
    related_tables = entry.get("related_tables", [])
    for table in related_tables:
        dataset_name, table_name = table.split(".")
        bq_entry_path = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@bigquery/entries/bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{dataset_name}/tables/{table_name}"
        
        base_link_id = f"lk-{term_id}-to-{dataset_name}-{table_name}".lower().replace("_", "-")
        link_id = base_link_id[:63].rstrip("-")
        
        print(f"  └─ 테이블 연동 중: {table} ...")
        if link_term_to_table(term_entry_path, bq_entry_path, link_id):
            print(f"     [SUCCESS] 연동 완료 ({link_id})")
            
        # 동의어들도 동일한 테이블과 연동 수립 (Related entries 에 노출하기 위함)
        synonyms = entry.get("synonyms", [])
        for idx, syn_name in enumerate(synonyms):
            syn_term_id = f"syn-{term_id}-{idx+1}"
            syn_entry_path = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@dataplex/entries/projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}/terms/{syn_term_id}"
            
            syn_link_id = f"lk-{syn_term_id}-to-{dataset_name}-{table_name}".lower().replace("_", "-")
            syn_link_id = syn_link_id[:63].rstrip("-")
            
            print(f"  └─ [동의어] 테이블 연동 중: {syn_name} ({syn_term_id}) <-> {table} ...")
            if link_term_to_table(syn_entry_path, bq_entry_path, syn_link_id):
                print(f"     [SUCCESS] 동의어 테이블 연동 완료 ({syn_link_id})")

    # B. 비즈니스 연관 용어 간 연동 (type = related)
    related_terms = entry.get("related_terms", [])
    for rel_id in related_terms:
        sorted_ids = sorted([term_id, rel_id])
        link_key = tuple(sorted_ids)
        if link_key in processed_links:
            continue
            
        rel_entry_path = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@dataplex/entries/projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}/terms/{rel_id}"
        base_link_id = f"lk-rel-{sorted_ids[0]}-to-{sorted_ids[1]}".lower().replace("_", "-")
        link_id = base_link_id[:63].rstrip("-")
        
        print(f"  └─ 연관 용어 연결 중: {term_id} <-> {rel_id} ...")
        if link_related_terms(term_entry_path, rel_entry_path, link_id):
            print(f"     [SUCCESS] 연관 완료 ({link_id})")
        processed_links.add(link_key)

    # C. 동의어 간 연동 (type = synonym)
    synonyms = entry.get("synonyms", [])
    for idx, syn_name in enumerate(synonyms):
        syn_term_id = f"syn-{term_id}-{idx+1}"
        syn_entry_path = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@dataplex/entries/projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}/terms/{syn_term_id}"
        
        sorted_ids = sorted([term_id, syn_term_id])
        link_key = tuple(sorted_ids)
        if link_key in processed_links:
            continue
            
        base_link_id = f"lk-syn-{sorted_ids[0]}-to-{sorted_ids[1]}".lower().replace("_", "-")
        link_id = base_link_id[:63].rstrip("-")
        
        print(f"  └─ 동의어 연결 중: {term_name} <-> {syn_name} ...")
        if link_synonym_terms(term_entry_path, syn_entry_path, link_id):
            print(f"     [SUCCESS] 동의어 연동 완료 ({link_id})")
        processed_links.add(link_key)

print("\n모든 비즈니스 용어집 계층 구조, 메타데이터 및 테이블 연동 정보가 Knowledge Catalog에 정상적으로 업로드되었습니다!")

## Step 8: 비즈니스 용어 Aspect(메타데이터) 설정

In [ ]:
# ----------------------------------------------------
# 모든 용어에 대해 aspect 설정
# ----------------------------------------------------
for entry in glossary_data:
    term_id = entry["term_id"]
    overview = entry.get("overview", "")
    synonyms = entry.get("synonyms", [])
    sql_mapping = entry.get("sql_mapping", None)
    
    # 1. 메인 용어 aspect 설정
    if overview:
        set_term_overview(term_id, overview)
            
    if sql_mapping:
        set_term_sql_mapping(term_id, sql_mapping)
            
    # 2. 동의어 aspect 설정 (메인 용어의 aspect 상속)
    for idx, syn_name in enumerate(synonyms):
        syn_term_id = f"syn-{term_id}-{idx+1}"
        
        if overview:
            set_term_overview(syn_term_id, overview)
                
        if sql_mapping:
            set_term_sql_mapping(syn_term_id, sql_mapping)

print("모든 비즈니스 용어에 대한 Aspect(개요 및 SQL 매핑) 설정이 완료되었습니다.")

## Step 9: Knowledge Catalog 용어집 등록 결과 조회 및 검증

In [ ]:
print("--- [등록된 카테고리 목록] ---")
!gcloud dataplex glossaries categories list --glossary={GLOSSARY_ID} --location={LOCATION} --format="table(displayName, name)"

print("\n--- [등록된 용어 목록] ---")
!gcloud dataplex glossaries terms list --glossary={GLOSSARY_ID} --location={LOCATION} --format="table(displayName, description, parent)"

## Appendix: 대화형 분석 검증 시나리오 및 SQL Mapping 가이드

등록된 Custom Aspect인 **`sql-mapping`**은 대화형 분석 에이전트(LLM)가 자연어를 SQL로 번역할 때 오차가 없도록 스니펫 및 수식을 대입해 주는 룰셋 역할을 합니다.

### 1. 주요 용어별 SQL 매핑 룰셋 조견표

| 용어 ID | 용어명 (디스플레이) | 매핑 종류 (`type`) | 타겟 테이블 (`table`) | 등록된 매핑 규칙 (`condition` / `expression` / `attribute`) |
| :--- | :--- | :--- | :--- | :--- |
| **`vip-customer`** | VIP 고객 | `SQL_Filter` | `thelook_ecommerce.users` | `condition`: `id IN (SELECT user_id FROM thelook_ecommerce.order_items GROUP BY user_id HAVING SUM(sale_price) >= 500 OR COUNT(DISTINCT order_id) >= 5)` |
| **`traffic-source`** | 유입 경로 | `Node_Attribute` | `thelook_ecommerce.users` | `attribute`: `traffic_source` |
| **`churned-customer`** | 이탈 고객 | `SQL_Filter` | `thelook_ecommerce.users` | `condition`: `id NOT IN (SELECT DISTINCT user_id FROM thelook_ecommerce.orders WHERE created_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 90 DAY))` |
| **`newly-registered-customer`** | 신규 가입 고객 | `SQL_Filter` | `thelook_ecommerce.users` | `condition`: `created_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)` |
| **`cart-abandonment-rate`** | 장바구니 포기율 | `Web_Analytics` | `thelook_ecommerce.events` | `expression`: `COUNT(DISTINCT CASE WHEN event_type = 'Cart' AND session_id NOT IN (SELECT DISTINCT session_id FROM thelook_ecommerce.events WHERE event_type = 'Purchase') THEN session_id END) / COUNT(DISTINCT CASE WHEN event_type = 'Cart' THEN session_id END) * 100` |
| **`profit-margin-rate`** | 마진율 | `SQL_Expression` | `thelook_ecommerce.products` | `expression`: `(retail_price - cost) / retail_price * 100` |
| **`returned-cancelled-items`** | 반품/취소 건 | `SQL_Filter` | `thelook_ecommerce.order_items` | `condition`: `status IN ('Returned', 'Cancelled')` |
| **`avg-shipping-lead-time`** | 평균 배송 소요 시간 | `SQL_Expression` | `thelook_ecommerce.orders` | `expression`: `AVG(TIMESTAMP_DIFF(delivered_at, shipped_at, DAY))` |
| **`return-rate`** | 취소/반품 비율 | `SQL_Expression` | `thelook_ecommerce.order_items` | `expression`: `COUNTIF(status IN ('Returned', 'Cancelled')) / COUNT(*) * 100` |

---

### 2. 에이전트 검증용 자연어 질문 및 예상 SQL (주요 검증 포인트 포함)

#### Q1. [우수 고객 집계] `"지난 달에 주문 이력이 존재하는 우수 고객은 총 몇 명인가요?"`
* **예상 결과 SQL:**
```sql
SELECT COUNT(DISTINCT u.id) AS vip_user_count
FROM `thelook_ecommerce.users` u
JOIN `thelook_ecommerce.orders` o ON u.id = o.user_id
WHERE u.id IN (
  SELECT user_id FROM `thelook_ecommerce.order_items` 
  GROUP BY user_id HAVING SUM(sale_price) >= 500 OR COUNT(DISTINCT order_id) >= 5
)
AND o.created_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY);
```
* **🔍 주요 검증 포인트 (Check Points):**
  1. **동의어 역맵핑:** 자연어의 **"우수 고객"**이 동의어 관계 명세를 통해 비즈니스 용어 **"VIP 고객" (ID: `vip-customer`)**으로 올바르게 해석되어 매핑되었는가?
  2. **필터 스니펫 주입:** **"VIP 고객" (ID: `vip-customer`)**에 등록된 `sql-mapping`의 `condition` 구문(`id IN (SELECT user_id...)`)이 WHERE 절 내부 조건식으로 완전하게 적용되었는가?
  3. **시간 필터 결합:** 최근 30일(지난 달)을 제한하는 `orders.created_at` 범위 필터가 AND 조건으로 누락 없이 작성되었는가?

#### Q2. [채널별 장바구니 포기율] `"유입 마케팅 채널별로 장바구니 이탈율을 계산해서 높은 순으로 정렬해줘."`
* **예상 결과 SQL:**
```sql
SELECT 
  u.traffic_source AS marketing_channel,
  COUNT(DISTINCT CASE WHEN e.event_type = 'Cart' AND e.session_id NOT IN (
    SELECT DISTINCT session_id FROM `thelook_ecommerce.events` WHERE event_type = 'Purchase'
  ) THEN e.session_id END) / 
  COUNT(DISTINCT CASE WHEN e.event_type = 'Cart' THEN e.session_id END) * 100 AS cart_abandonment_rate
FROM `thelook_ecommerce.events` e
JOIN `thelook_ecommerce.users` u ON e.user_id = u.id
GROUP BY marketing_channel
ORDER BY cart_abandonment_rate DESC;
```
* **🔍 주요 검증 포인트 (Check Points):**
  1. **마케팅 채널 속성 매핑:** 자연어의 **"마케팅 채널"**이 비즈니스 용어 **"유입 경로" (ID: `traffic-source`)**로 매핑되어, 속성 정보(`users.traffic_source`)를 SELECT 및 GROUP BY 절에 정상 반영했는가?
  2. **수식 형태 메트릭 주입:** **"장바구니 이탈율"**이 비즈니스 용어 **"장바구니 포기율" (ID: `cart-abandonment-rate`)**의 `expression`에 적힌 복잡한 비즈니스 로그 공식으로 정확하게 치환되었는가?
  3. **정렬 및 그룹화:** 마케팅 채널을 기준으로 집계하고 계산된 포기율의 내림차순(`DESC`) 정렬이 제대로 구현되었는가?

#### Q3. [수익성/품질 분석] `"여성 의류 중에서 상품 마진이 40% 이상인 제품들의 환불률을 브랜드별로 계산해줘."`
* **예상 결과 SQL:**
```sql
SELECT 
  p.brand,
  COUNTIF(oi.status IN ('Returned', 'Cancelled')) / COUNT(*) * 100 AS refund_rate
FROM `thelook_ecommerce.order_items` oi
JOIN `thelook_ecommerce.products` p ON oi.product_id = p.id
WHERE p.category = 'Women'
  AND ((p.retail_price - p.cost) / p.retail_price * 100) >= 40
GROUP BY p.brand
ORDER BY refund_rate DESC;
```
* **🔍 주요 검증 포인트 (Check Points):**
  1. **마진율 공식 주입:** 자연어 **"상품 마진"**이 비즈니스 용어 **"마진율" (ID: `profit-margin-rate`)**의 `expression`인 `(retail_price - cost) / retail_price * 100` 수식으로 대입되고, 조건 필터(`>= 40`)와 결합되었는가?
  2. **환불률 공식 주입:** 자연어 **"환불률"**이 비즈니스 용어 **"취소/반품 비율" (ID: `return-rate`)**의 `expression` 공식인 `COUNTIF(status IN ('Returned', 'Cancelled')) / COUNT(*) * 100`으로 완벽히 적용되었는가?
  3. **다중 관계 조인:** `products`와 `order_items` 테이블 간의 상품 ID 조인 결합 관계가 어긋남 없이 연결되었는가?

#### Q4. [휴면고객 유입 채널] `"휴면 고객들이 가장 많이 유입되었던 가입 경로 3가지는 무엇인가요?"`
* **예상 결과 SQL:**
```sql
SELECT 
  traffic_source AS signup_channel,
  COUNT(*) AS user_count
FROM `thelook_ecommerce.users`
WHERE id NOT IN (
  SELECT DISTINCT user_id FROM `thelook_ecommerce.orders` WHERE created_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 90 DAY)
)
GROUP BY signup_channel
ORDER BY user_count DESC
LIMIT 3;
```
* **🔍 주요 검증 포인트 (Check Points):**
  1. **부정 필터 조건 반영:** 자연어 **"휴면 고객"**이 비즈니스 용어 **"이탈 고객" (ID: `churned-customer`)**으로 대응되어 최근 90일 미주문 조건인 부정 필터문(`id NOT IN (SELECT user_id...)`)이 정확히 기술되었는가?
  2. **가입 경로 속성 매핑:** **"가입 경로"**가 비즈니스 용어 **"유입 경로" (ID: `traffic-source`)**의 매핑 컬럼인 `traffic_source`로 치환되었는가?
  3. **제약 조건과 정렬:** 가입 경로별 집계 결과에 대해 내림차순 정렬(`ORDER BY user_count DESC`)과 반환 레코드 수 제한(`LIMIT 3`) 조건이 정상 추가되었는가?